# 09 — Production patterns: preflight, errors, retries, tuning

When you move beyond the quickstart into real workloads, four small things make the difference between a notebook that runs once and a pipeline that survives in production:

1. **Pre-flight** — catch low-sun configurations before paying for tiles whose results won't have enough geometric context.
2. **Big-payload errors** — dense polygons or full-year weather take a presign → S3 → `$ref` envelope path. The receiving service can fail with typed `BigPayload*` errors that you can dispatch on by `.code`.
3. **Retry-After** — rate-limit responses carry the wait hint in `JobSubmitError.retry_after`. Honor it instead of guessing.
4. **Threshold tuning** — when 5 MiB is the wrong cutoff for your shape distribution, change it via env var.

This notebook is mostly **patterns**, not live API calls — the failure cases (presign failure, S3 PUT failure, rate limit, ref expiry) are infrastructure events you can't reliably trigger from polygon geometry. The pre-flight call in section 1 *is* live (it runs entirely client-side in microseconds).

In [1]:
from dotenv import load_dotenv

load_dotenv()

import os
import time

from infrared_sdk import (
    InfraredClient,  # noqa: F401  # used in commented patterns below
    WindModelRequest,
    # Async-job error hierarchy
    JobSubmitError,
    # Big-payload error hierarchy
    BigPayloadError,
    BigPayloadFetchError,
    BigPayloadPresignError,
    BigPayloadUploadError,
    RefExpiredRetryExhausted,
    # Weather/utilities-service error (with retry_after; auto-retry on 429/5xx since v0.4.9)
    WeatherServiceError,  # noqa: F401  # documentation-only import — see Section 3
    # Pre-flight diagnostics
    estimate_sun_context_loss,
    SUN_CONTEXT_SEVERITY_LEVELS,
)
from infrared_sdk.analyses.types import AnalysesName

## 1. Pre-flight: catch low-sun runs before submitting

Solar / daylight / direct-sun-hours / thermal-comfort analyses use the **solar tiling config** (128 m geometry buffer per tile). When the sun is low — winter mornings or evenings at high latitudes — buildings near the tile edge cast shadows longer than the buffer can hold, so the simulation sees only part of the shadow context. The grid still comes back; it's just quietly less accurate.

`estimate_sun_context_loss(...)` answers "is this configuration in trouble?" entirely client-side, in microseconds, before you pay for any tile. The return is a `SunContextResult` TypedDict with:

- **`severity`** — one of `("info", "ok", "marginal", "warning", "critical")`
- **`message`** — human-readable explanation suitable for a banner
- numeric fields: `min_sun_elevation_deg`, `buffer_breakeven_height_m`, `shadow_at_typical_building_m`, etc.

It **never raises** — bad input comes back as `severity="info"` with a diagnostic message — so it's safe to call unconditionally before any submission.

In [2]:
# Berlin, winter mornings, mid-rise to occasional high-rise mix.
# At 8–11 in December the sun barely clears 15° — long shadows.
result = estimate_sun_context_loss(
    lat=52.5219,
    lon=13.4132,
    start_month=12,
    start_day=1,
    start_hour=8,
    end_month=12,
    end_day=31,
    end_hour=11,
    buffer_m=128.0,  # current SOLAR_TILING_CONFIG margin
    typical_building_height_m=25.0,
    max_building_height_m=80.0,
)

print(f"severity              {result['severity']}")
print(f"min sun elevation     {result['min_sun_elevation_deg']:.1f}°")
print(f"pattern               {result['pattern']}")
print(f"buffer covers up to   {result['buffer_breakeven_height_m']:.0f} m buildings")
print()
print(result["message"])

# Decision: gate the submission on severity
if result["severity"] in ("warning", "critical"):
    print("\n→ Reconsider time window or restrict to higher-sun hours.")
else:
    print("\n→ Safe to proceed.")

print(f"\n(known severities: {SUN_CONTEXT_SEVERITY_LEVELS})")

severity              critical
min sun elevation     3.9°
pattern               Northern hemisphere, winter, late morning
buffer covers up to   9 m buildings

Context buffer is severely insufficient. Sun elevation reaches 4° on 12-21 09:00 (Northern hemisphere, winter, late morning). The 128 m buffer covers only buildings up to 8.7 m, well below the configured 25 m typical / 80 m max. Most shadow context is missing at tile edges; results are reliable only deep in tile interiors. Recommend either restricting the time window above the horizon or increasing the geometry-context size.

→ Reconsider time window or restrict to higher-sun hours.

(known severities: ('info', 'ok', 'marginal', 'warning', 'critical'))


## 2. Big-payload errors

Above 5 MiB (raw `json.dumps` size) the SDK transparently switches to the `$ref` envelope path: zip the payload → POST presign → S3 PUT → POST `{"$ref": <url>}` to the original endpoint. Three things can fail at infrastructure level:

| Exception | What happened |
|---|---|
| `BigPayloadPresignError`   | The gateway refused to issue a presigned upload URL. |
| `BigPayloadUploadError`    | The S3 PUT failed (signature mismatch, transient 5xx). |
| `BigPayloadFetchError`     | The receiving service couldn't fetch the `$ref` — typed `REF_*` code on `.code`. |
| `RefExpiredRetryExhausted` | Sub-case of fetch: the upload's presigned GET URL expired and the retry budget ran out. Carries `.code = "REF_EXPIRED"`. |

All four inherit from `BigPayloadError`, so a single `except BigPayloadError` catches everything if you don't need to dispatch. When you do — for example, retry on `REF_EXPIRED` but fail fast on `REF_TOO_LARGE` — match on `.code`.

Order `except` clauses **most specific first** (`RefExpiredRetryExhausted` before its parent `BigPayloadFetchError`, all four before `BigPayloadError`). Every exception also carries `.status_code` and `.response_body` as keyword-only attributes — these never land in `args[0]`, so they don't leak into Sentry's default capture.

In [3]:
POLYGON = {
    "type": "Polygon",
    "coordinates": [
        [
            [11.5712, 48.1360],
            [11.5798, 48.1360],
            [11.5798, 48.1388],
            [11.5712, 48.1388],
            [11.5712, 48.1360],
        ]
    ],
}
WIND = WindModelRequest(
    analysis_type=AnalysesName.wind_speed,
    wind_speed=5,
    wind_direction=270,
)


def run_with_bigpayload_handling(client, payload, polygon, *, max_attempts=3):
    """Submit, wait, return AreaResult.

    Retries once on REF_EXPIRED (a re-submit generates a fresh presigned URL).
    Surfaces presign / upload / other-fetch errors immediately — none are
    retriable from the client.
    """
    for attempt in range(1, max_attempts + 1):
        try:
            return client.run_area_and_wait(payload, polygon)
        except RefExpiredRetryExhausted as e:
            if attempt == max_attempts:
                print(f"[abort] {e.code}: exhausted {max_attempts} attempts")
                raise
            print(f"[retry] {e.code} on attempt {attempt} — resubmitting")
            continue
        except BigPayloadFetchError as e:
            # REF_TOO_LARGE, REF_INVALID_ENVELOPE, REF_HOST_NOT_ALLOWED,
            # REF_CONTENT_TYPE_REJECTED, REF_DECODE_FAILED, REF_FETCH_TIMEOUT.
            # None are retriable from the client side.
            print(f"[abort] {e.code}: {e} (HTTP {e.status_code})")
            raise
        except BigPayloadPresignError as e:
            print(f"[abort] presign failed (HTTP {e.status_code}): {e}")
            raise
        except BigPayloadUploadError as e:
            print(f"[abort] S3 upload failed (HTTP {e.status_code}): {e}")
            raise
        except BigPayloadError as e:
            # Catch-all for any future big-payload subclass we don't yet match.
            print(f"[abort] big-payload error: {e}")
            raise


# Usage (commented to avoid charging a tile run on smoke-test):
# with InfraredClient() as client:
#     result = run_with_bigpayload_handling(client, WIND, POLYGON)

print("Pattern defined. Uncomment the `with InfraredClient()` block to run live.")

Pattern defined. Uncomment the `with InfraredClient()` block to run live.


## 3. Rate limits: honor `Retry-After`

When the API rate-limits a submission, the response carries a `Retry-After` header in delta-seconds. The SDK parses it into `.retry_after` on the relevant typed exceptions:

- `JobSubmitError.retry_after` — on submit
- `JobPollError.retry_after` — on status poll
- `ResultsDownloadError.retry_after` — on result fetch
- `WeatherServiceError.retry_after` — on weather / utilities endpoints (since v0.4.9). The SDK auto-retries 3× internally honoring `Retry-After`; this attribute is populated from the final failing response so callers can implement outer backoff once the SDK's budget is exhausted.

Honor the hint instead of guessing. Naive `sleep(60)` either wastes budget when the gate clears quickly or fails again when it doesn't.

Note: HTTP-date format is intentionally *not* parsed — the server side always uses delta-seconds for rate limiting, so `.retry_after` is either a float or `None`. Always coalesce to a sensible floor before sleeping.

In [4]:
def submit_with_retry_after(client, payload, polygon, *, max_attempts=4):
    """Submit; back off on 429 + Retry-After; surface other failures."""
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return client.run_area(payload, polygon)
        except JobSubmitError as e:
            last_exc = e
            # Real failure (not a rate limit) — don't retry.
            if e.status_code != 429 or e.retry_after is None:
                raise
            wait = max(e.retry_after, 1.0)
            print(f"[rate-limited] attempt {attempt}: waiting {wait:.1f}s")
            time.sleep(wait)
    assert last_exc is not None
    raise last_exc


print("Pattern defined. Wrap any submit call with this under bursty load.")

Pattern defined. Wrap any submit call with this under bursty load.


## 4. Tuning the big-payload threshold

The 5 MiB cutoff is conservative. Two reasons to change it:

- **Lower it** if your typical payload sits just below 5 MiB and you'd rather take the envelope path consistently than oscillate around the boundary.
- **Raise it** if you're submitting many small tiles and want to skip the envelope round-trip entirely until payloads get genuinely large.

Set `INFRARED_BIG_PAYLOADS_THRESHOLD_BYTES` (raw `json.dumps` size, in bytes) before constructing the client. The check is **strict greater-than**, so any value above the threshold takes the envelope path.

There is also a kill switch — `INFRARED_BIG_PAYLOADS_ENABLED=false` forces every request through the inline path regardless of size. Use only for debugging; the gateway has its own ceiling and will reject oversize inline payloads with 413.

In [5]:
# Lower the threshold to 1 MiB (so 1.5 MiB payloads take the envelope path).
os.environ["INFRARED_BIG_PAYLOADS_THRESHOLD_BYTES"] = str(1 * 1024 * 1024)

# Kill switch (commented — debugging only):
# os.environ["INFRARED_BIG_PAYLOADS_ENABLED"] = "false"

# The SDK reads these env vars per request, so no client rebuild is needed.
print(
    f"BIG_PAYLOADS_THRESHOLD = {os.environ['INFRARED_BIG_PAYLOADS_THRESHOLD_BYTES']} bytes"
)
print(
    f"BIG_PAYLOADS_ENABLED   = {os.environ.get('INFRARED_BIG_PAYLOADS_ENABLED', '(default: true)')}"
)

# Reset for cleanliness.
del os.environ["INFRARED_BIG_PAYLOADS_THRESHOLD_BYTES"]

BIG_PAYLOADS_THRESHOLD = 1048576 bytes
BIG_PAYLOADS_ENABLED   = (default: true)


## Where to go from here

- **Notebook 03** — TimePeriod + weather files. Run the pre-flight check alongside `client.weather.filter_weather_data()` before queueing a long solar run.
- **Notebook 07** — async + webhooks. Combine the `Retry-After` pattern above with `client.run_area` / `check_area_state` / `merge_area_jobs` for resilient long-running pipelines.
- **Notebook 08** — wind tile merge strategies (`default`, `directional`, `directional_blend`) with full side-by-side visualisation.
- **CLAUDE.md** in the repo root — `Conventions` section has the full big-payload contract, the kwarg-only exception attribute convention, and the tiling-config invariants.